# Man-in-the-Middle — Hijack

Here the **attacker itself** performs the intervention, not the GCS. The MITM
runs a real `Intervention` — the same trigger + guided plan a `gcs.intervene(...)`
call would use — from its interposed position. It watches the telemetry it is
relaying and, once the drone reaches a chosen mission point
(`MISSION_CURRENT.seq >= 4`), it drives the vehicle through the attacker's
waypoints: switching it to GUIDED and repositioning it. The injected commands
are spoofed to look like they came from the GCS (sysid 255), so the onboard
Logic forwards them to the flight controller as legitimate.

The only thing that differs from a GCS intervention is *where* it runs: the
exact same `InterventionRunner` drives it, bound to the MITM's command link
instead of the GCS's. This is a *visible* hijack — telemetry still flows to the
GCS, so the operator watches the drone get redirected — and the GCS itself
sends no commands (no `gcs.intervene(...)` call is made).

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import SimProcess
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import InterventionStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, -20, 0, 0)
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

mission_wps = ENU.list(
    [(0, 0, 0), (0, 0, cruise_alt), (0, 20, cruise_alt), (0, 40, cruise_alt)]
)

# Attacker's destination, as an ENU relative to the run origin — the
# InterventionPlan converts it to geodetic when it binds inside the MITM.
hijack_target = ENU(x=10, y=0, z=cruise_alt)
print(f"Hijack target (ENU): {hijack_target}")

## Vehicle + MITM hijack

In [ ]:
mission_path = DATA_PATH / "missions" / "gcs_intervention.waypoints"


plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=str(mission_path),
    firmware=model.firmware,
)


# The attacker runs a GCS-style Intervention from the MITM position: the same
# trigger + guided plan a `gcs.intervene(...)` call would build, only injected
# by the man-in-the-middle and spoofed to look like the GCS.
hijack = InterventionStrategy(
    Intervention(
        trigger=MissionTrigger(seq=4),
        plan=InterventionPlan.from_relative_path(
            relative_path=[hijack_target],
            enu_origin=enu_origin,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    )
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcss=[SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")],
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
    mitm=hijack,
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)

target_marker = GazMarker(
    name="hijack_target",
    group="targets",
    pos=hijack_target,
    color=Color.RED,
)
gaz.markers.append(target_marker)
gaz.markers.append(origin_marker)


## Oracle

In [ ]:
orac = Oracle()
orac.add_vehicle(vehicle)

##  Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    verbose=2,
    terminals=[SimProcess.MITM, SimProcess.LOGIC, SimProcess.GCS],
    speedup=1,
)
simulator.preview()

In [ ]:
simulator.run(timeout=90)


## What to observe

- The drone flies north, then **turns west** mid-mission — driven entirely by
  the man-in-the-middle, with no command from the GCS.
- `simulator/logs/mitm/mitm_1.log` — `MITM intervention: taking over vehicle 1`
  (the shared `InterventionRunner`, now running inside the MITM).
- `simulator/data/mitm_cmd/` — the command traffic the attacker's intervention
  emitted (separate from the GCS's own `gcs_cmd/` log).
- `simulator/logs/logics/logic_1.log` — `GCS→SITL forwarding SET_MODE` /
  `COMMAND_INT` (Logic forwards the spoofed commands to the flight controller).
- `simulator/logs/GCSs/GCS_BLUE_*.log` — **no** `GCS intervention` line; the
  redirect did not originate from the GCS.